# EcoPackAI

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import openpyxl
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib

## Loading the Dataset

In [2]:
df = pd.read_csv("dataset/EcoPackAI_Final_7.csv")

## Data Preprocessing

In [3]:
df_db = df.rename(columns={
    "Material_Type": "material_type",
    "Product_Type": "product_type",
    "Industry": "industry",
    "Strength (1-10)": "strength",
    "Weight Capacity (kg)": "weight_capacity",
    "Biodegradability Score (1-10)": "biodegradability_score",
    "Recyclability (%)": "recyclability",
    "Cost (USD)": "cost",
    "CO2 Emission (kg CO2/kg)": "co2_emission"
})

df_db.to_csv("dataset/ecoPackAI_db.csv", index=False)

In [4]:
print("Preview of Dataset")
display(df.head())

Preview of Dataset


,Material_Type,Product_Type,Industry,Strength (1-10),Weight Capacity (kg),Biodegradability Score (1-10),Recyclability (%),Cost (USD),CO2 Emission (kg CO2/kg)
0,Cardboard,Cereal Box,Food & Beverage,9,0.54,9,86.79,1.51,0.70
1,Molded Pulp,Smartwatch Box,Electronics,7,0.44,10,78.41,3.47,0.95
2,Bagasse Fiber,Shopping Bag,Retail,8,1.35,8,72.89,3.15,1.34
3,Bioplastic (PLA),Lipstick Tube,Cosmetics,3,4.63,9,62.50,3.76,0.67
4,PET Plastic,Beverage Can,Food & Beverage,6,4.56,2,55.96,2.49,4.73


In [5]:
# Info about columns and datatypes
print("\nDataset Info:")
print(df.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19600 entries, 0 to 19599
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Material_Type                  19600 non-null  object 
 1   Product_Type                   19600 non-null  object 
 2   Industry                       19600 non-null  object 
 3   Strength (1-10)                19600 non-null  int64  
 4   Weight Capacity (kg)           19600 non-null  float64
 5   Biodegradability Score (1-10)  19600 non-null  int64  
 6   Recyclability (%)              19600 non-null  float64
 7   Cost (USD)                     19600 non-null  float64
 8   CO2 Emission (kg CO2/kg)       19600 non-null  float64
dtypes: float64(4), int64(2), object(3)
memory usage: 1.3+ MB
None


In [6]:
# Descriptive statistics
print("\nDescriptive Statistics:")
display(df.describe())


Descriptive Statistics:


,Strength (1-10),Weight Capacity (kg),Biodegradability Score (1-10),Recyclability (%),Cost (USD),CO2 Emission (kg CO2/kg)
count,19600.000000,19600.000000,19600.000000,19600.000000,19600.000000,19600.000000
mean,6.531224,6.822372,5.104541,75.292684,2.618288,2.273639
std,2.273988,12.347479,3.515281,10.113407,1.146973,1.422207
min,3.000000,0.050000,1.000000,50.010000,1.000000,0.300000
25%,5.000000,0.450000,2.000000,69.390000,1.890000,1.110000
50%,7.000000,1.570000,3.000000,75.830000,2.455000,1.870000
75%,8.000000,7.120000,9.000000,83.042500,3.010000,3.240000
max,10.000000,99.870000,10.000000,95.000000,9.980000,7.980000


In [7]:
# Missing values check
print("\nMissing Values per Column:")
print(df.isnull().sum())


Missing Values per Column:
Material_Type                    0
Product_Type                     0
Industry                         0
Strength (1-10)                  0
Weight Capacity (kg)             0
Biodegradability Score (1-10)    0
Recyclability (%)                0
Cost (USD)                       0
CO2 Emission (kg CO2/kg)         0
dtype: int64


In [8]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (19600, 9)


In [9]:
#Encoding Categorical Values
df = pd.get_dummies(
    df,
    columns=["Material_Type", "Product_Type", "Industry"],
    drop_first=True
)

## Random Forest (Cost Prediction)

In [10]:
# Features and target selection for Random Forest
X_cost = df.drop("Cost (USD)", axis=1)
y_cost = df["Cost (USD)"]

In [11]:
#train , test set split
X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_cost,
    y_cost,
    test_size=0.2,
    random_state=42
)
X_val_c, X_test_c, y_val_c, y_test_c = train_test_split(
    X_temp_c,
    y_temp_c,
    test_size=0.5,
    random_state=42
)
print("Train size:", X_train_c.shape)
print("Validation size:", X_val_c.shape)
print("Test size:", X_test_c.shape)

Train size: (15680, 103)
Validation size: (1960, 103)
Test size: (1960, 103)


In [12]:
# saving column names 
cost_feature_columns = X_train_c.columns
joblib.dump(cost_feature_columns, "trained_models/cost_feature_columns.pkl")

['trained_models/cost_feature_columns.pkl']

In [13]:
# feature scaling Random forest
scaler = StandardScaler()

numerical_cols = [
    "Strength (1-10)",
    "Weight Capacity (kg)",
    "Biodegradability Score (1-10)",
    "Recyclability (%)"
]

X_train_c[numerical_cols] = scaler.fit_transform(X_train_c[numerical_cols])
X_val_c[numerical_cols] = scaler.transform(X_val_c[numerical_cols])
X_test_c[numerical_cols] = scaler.transform(X_test_c[numerical_cols])

In [14]:
# Training Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42
)

rf_model.fit(X_train_c, y_train_c)

,n_estimators,200
,criterion,'squared_error'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
# Validation Evaluation Random Forest
y_val_pred = rf_model.predict(X_val_c)

print("Validation Results - Random Forest")

print("MAE:", mean_absolute_error(y_val_c, y_val_pred))
print("RMSE:", root_mean_squared_error(y_val_c, y_val_pred))
print("R2:", r2_score(y_val_c, y_val_pred))

Validation Results - Random Forest
MAE: 0.5574694502269048
RMSE: 0.6697614566351584
R2: 0.6481920501492607


In [16]:
# Test evaluation Random Forest
y_test_pred = rf_model.predict(X_test_c)

print("Test Results - Random Forest")

print("MAE:", mean_absolute_error(y_test_c, y_test_pred))
print("RMSE:", root_mean_squared_error(y_test_c, y_test_pred))
print("R2:", r2_score(y_test_c, y_test_pred))

Test Results - Random Forest
MAE: 0.556790454302056
RMSE: 0.6704166150826619
R2: 0.6642296438015891


## XGBoost (Co2 Prediction)

In [17]:
# feature and target selection for XGBoost
X_co2 = df.drop("CO2 Emission (kg CO2/kg)", axis=1)
y_co2 = df["CO2 Emission (kg CO2/kg)"]

In [18]:
# train test validation split XGBoost
X_train_co2, X_temp_co2, y_train_co2, y_temp_co2 = train_test_split(
    X_co2,
    y_co2,
    test_size=0.2,
    random_state=42
)

X_val_co2, X_test_co2, y_val_co2, y_test_co2 = train_test_split(
    X_temp_co2,
    y_temp_co2,
    test_size=0.5,
    random_state=42
)

In [19]:
# saving column name
co2_feature_columns = X_train_co2.columns
joblib.dump(co2_feature_columns, "trained_models/co2_feature_columns.pkl")

['trained_models/co2_feature_columns.pkl']

In [20]:
# feature scaling XGBoost
scaler = StandardScaler()

X_train_co2[numerical_cols] = scaler.fit_transform(X_train_co2[numerical_cols])
X_val_co2[numerical_cols] = scaler.transform(X_val_co2[numerical_cols])
X_test_co2[numerical_cols] = scaler.transform(X_test_co2[numerical_cols])

In [21]:
# training XGBoost
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train_co2, y_train_co2)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [22]:
# validation evaluation
y_val_pred_co2 = xgb_model.predict(X_val_co2)

print("Validation Results - XGBoost")
print("MAE:", mean_absolute_error(y_val_co2, y_val_pred_co2))
print("RMSE:", root_mean_squared_error(y_val_co2, y_val_pred_co2))
print("R2:", r2_score(y_val_co2, y_val_pred_co2))

Validation Results - XGBoost
MAE: 0.46894781407409786
RMSE: 0.5861446756463773
R2: 0.8226347081363046


In [23]:
# test evaluation
y_test_pred_co2 = xgb_model.predict(X_test_co2)

print("Test Results - XGBoost")
print("MAE:", mean_absolute_error(y_test_co2, y_test_pred_co2))
print("RMSE:", root_mean_squared_error(y_test_co2, y_test_pred_co2))
print("R2:", r2_score(y_test_co2, y_test_pred_co2))

Test Results - XGBoost
MAE: 0.46232215181297187
RMSE: 0.5829662813749132
R2: 0.8353484614970931


In [24]:
#processed file
df.to_csv("dataset/EcoPackAI_Preprocessed.csv", index=False)

In [25]:
# saving the trained model
pipeline_cost = Pipeline([
    ("scaler", StandardScaler()),
    ("model", rf_model)
])

pipeline_co2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", xgb_model)
])

joblib.dump(rf_model, "trained_models/cost_model.pkl")
joblib.dump(xgb_model, "trained_models/co2_model.pkl")

['trained_models/co2_model.pkl']